# CGAS recipe LoRA pilot

Select **Runtime > Change runtime type > T4 GPU**, then run these cells in order.
This notebook fine-tunes Qwen2.5-0.5B-Instruct with real LoRA adapter updates.
The labels come from the recipe app's fixed rules and are **unreviewed weak supervision**.
The experiment measures policy imitation, not independent taste or recommendation quality.

The notebook contains its scripts and ingredient snapshot. No second upload is required.
After it completes, download the output ZIP before ending the Colab session.

Do not put future-work training claims into the past tense until this run finishes.
Sources: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct),
[Hugging Face PEFT](https://huggingface.co/docs/peft/quicktour).


## 1. Check that a GPU is available

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, reconnect, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


## 2. Create the bundled training files

In [ ]:
import json
from pathlib import Path
TASK = Path('/content/recipe-lora')
BUNDLE = json.loads('{"ml/policy.py": "\\"\\"\\"Weak supervision for a LoRA pilot. These labels are NOT expert ground truth.\\"\\"\\"\\nfrom __future__ import annotations\\nimport hashlib\\nimport json\\nimport math\\nimport random\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parent\\nSNAPSHOT = ROOT / \\"food_snapshot.json\\"\\nSYSTEM = (\\n    \\"Select one ingredient replacement from the supplied candidates. \\"\\n    \\"Only candidates with eligible=true may be selected. \\"\\n    \\"Among eligible candidates maximize 0.35*F+0.30*N+0.25*C+0.10*R. \\"\\n    \\"F is flavor similarity, N nutrition, C cuisine compatibility, R category compatibility. \\"\\n    \\"Return ONLY JSON with exactly one key candidate_id, containing a candidate id, \\"\\n    \\"or null when no candidate is eligible. These are illustrative rule scores.\\"\\n)\\nEXCLUDED = {\\"pecorino\\"}  # Supplied per100g entry encodes a half portion. Await source audit.\\nCATEGORY = {\\n    \\"fat \\u00b7 dairy\\": {\\"fat \\u00b7 dairy\\": 1, \\"fat \\u00b7 plant\\": .75, \\"fat \\u00b7 fruit\\": .65, \\"dairy\\": .7},\\n    \\"dairy\\": {\\"dairy\\": 1, \\"plant cream\\": .8, \\"dairy \\u00b7 fermented\\": .85},\\n    \\"dairy \\u00b7 fermented\\": {\\"dairy \\u00b7 fermented\\": 1, \\"dairy\\": .85, \\"plant cream\\": .7},\\n    \\"cured meat\\": {\\"cured meat\\": 1, \\"vegetable \\u00b7 fungi\\": .55, \\"meat\\": .7},\\n    \\"meat\\": {\\"meat\\": 1, \\"meat \\u00b7 poultry\\": .85, \\"plant protein\\": .6, \\"legume\\": .55},\\n    \\"cheese \\u00b7 aged\\": {\\"cheese \\u00b7 aged\\": 1, \\"seasoning\\": .55},\\n    \\"sauce \\u00b7 emulsion\\": {\\"sauce \\u00b7 emulsion\\": 1},\\n    \\"grain \\u00b7 baked\\": {\\"grain \\u00b7 baked\\": 1, \\"legume\\": .6},\\n    \\"fat \\u00b7 infused\\": {\\"fat \\u00b7 infused\\": 1, \\"fat \\u00b7 plant\\": .7},\\n}\\n\\ndef read_snapshot():\\n    return json.loads(SNAPSHOT.read_text(encoding=\\"utf-8\\"))\\n\\ndef snapshot_hash():\\n    return hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest()\\n\\ndef cosine(a, b):\\n    denom = math.sqrt(sum(x*x for x in a) * sum(x*x for x in b))\\n    return sum(x*y for x,y in zip(a,b)) / denom if denom else 0.0\\n\\ndef choose_teacher(candidates):\\n    eligible = [c for c in candidates if c[\\"eligible\\"]]\\n    # Deterministic tie-breaking avoids order-dependent labels.\\n    return min(eligible, key=lambda c: (-c[\\"score\\"], c[\\"id\\"]))[\\"id\\"] if eligible else None\\n\\ndef build_case(recipe_id, ingredient_id, goal_id, variant=0, seed=42):\\n    data = read_snapshot()\\n    recipe = next(r for r in data[\\"recipes\\"] if r[\\"id\\"] == recipe_id)\\n    item = next(i for i in recipe[\\"items\\"] if i[\\"ingredientId\\"] == ingredient_id and i.get(\\"replaceable\\"))\\n    goal = next(g for g in data[\\"goals\\"] if g[\\"id\\"] == goal_id)\\n    orig = data[\\"ingredients\\"][ingredient_id]\\n    original = orig[\\"per100g\\"][goal_id]\\n    out = []\\n    for cid in item[\\"replaceable\\"]:\\n        if cid in EXCLUDED:\\n            continue\\n        cand = data[\\"ingredients\\"][cid]\\n        value = cand[\\"per100g\\"][goal_id]\\n        change = (original-value) if goal[\\"direction\\"] == \\"min\\" else (value-original)\\n        eligible = change > (original*.02 if original > 0 else 1e-9)\\n        # Explicit 1 g/kcal normalization floor avoids 1e-6 blow-ups for zero nutrients.\\n        raw = max(-20., min(20., change/max(abs(original),1.)))\\n        features = {\\n            \\"F\\": round(cosine(orig[\\"flavor\\"],cand[\\"flavor\\"]),4),\\n            \\"N\\": round(1/(1+math.exp(-3*raw)),4),\\n            \\"C\\": round(cand[\\"cuisine\\"].get(recipe[\\"cuisine\\"],.1),4),\\n            \\"R\\": round(CATEGORY.get(orig[\\"category\\"],{}).get(cand[\\"category\\"],.45),4),\\n        }\\n        score = sum(features[k]*w for k,w in {\\"F\\":.35,\\"N\\":.30,\\"C\\":.25,\\"R\\":.10}.items())\\n        out.append({\\"id\\":cid,\\"name\\":cand[\\"name\\"],**features,\\"eligible\\":eligible,\\"score\\":score})\\n    target = choose_teacher(out)\\n    rng = random.Random(f\\"{seed}:{recipe_id}:{ingredient_id}:{goal_id}:{variant}\\")\\n    if variant:\\n        rng.shuffle(out)\\n    # Variants shuffle ordering only. They are NOT independent new recipes.\\n    payload = {\\"recipe\\":recipe[\\"name\\"],\\"cuisine\\":recipe[\\"cuisine\\"],\\n               \\"original\\":orig[\\"name\\"],\\"grams\\":item[\\"grams\\"],\\"goal\\":goal[\\"label\\"],\\n               \\"candidates\\":[{k:v for k,v in c.items() if k!=\\"score\\"} for c in out]}\\n    completion = json.dumps({\\"candidate_id\\":target},separators=(\\",\\",\\":\\"))\\n    return {\\"query_id\\":f\\"{recipe_id}/{ingredient_id}/{goal_id}\\",\\"recipe_family\\":recipe_id,\\n            \\"recipe_id\\":recipe_id,\\"ingredient_id\\":ingredient_id,\\"goal_id\\":goal_id,\\n            \\"variant\\":variant,\\"messages\\":[{\\"role\\":\\"system\\",\\"content\\":SYSTEM},\\n            {\\"role\\":\\"user\\",\\"content\\":json.dumps(payload,separators=(\\",\\",\\":\\"))}],\\n            \\"completion\\":completion,\\"target\\":target,\\"candidates\\":out,\\n            \\"label_source\\":\\"fixed_rule_teacher_v2\\",\\"review_status\\":\\"unreviewed\\",\\n            \\"teacher_policy\\":\\"fixed_weights_eligible_first_floor_1\\"}\\n\\ndef make_dataset(directory):\\n    directory = Path(directory)\\n    directory.mkdir(parents=True,exist_ok=True)\\n    family_split={\\"butter-chicken\\":\\"train\\",\\"mapo-tofu\\":\\"train\\",\\"caesar\\":\\"train\\",\\n                  \\"stroganoff\\":\\"validation\\",\\"carbonara\\":\\"test\\"}\\n    rows={k:[] for k in [\\"train\\",\\"validation\\",\\"test\\"]}\\n    canonical=[]\\n    data=read_snapshot()\\n    for recipe in data[\\"recipes\\"]:\\n        split=family_split[recipe[\\"id\\"]]\\n        for item in recipe[\\"items\\"]:\\n            if not item.get(\\"replaceable\\"):continue\\n            for goal in data[\\"goals\\"]:\\n                canonical.append(build_case(recipe[\\"id\\"],item[\\"ingredientId\\"],goal[\\"id\\"]))\\n                for variant in range(8 if split==\\"train\\" else 3):\\n                    rows[split].append(build_case(recipe[\\"id\\"],item[\\"ingredientId\\"],goal[\\"id\\"],variant))\\n    for split, records in rows.items():\\n        (directory/f\\"{split}.jsonl\\").write_text(\\"\\".join(json.dumps(x)+\\"\\\\n\\" for x in records),encoding=\\"utf-8\\")\\n    (directory/\\"canonical.jsonl\\").write_text(\\"\\".join(json.dumps(x)+\\"\\\\n\\" for x in canonical),encoding=\\"utf-8\\")\\n    manifest={\\"snapshot_sha256\\":snapshot_hash(),\\"label_source\\":\\"fixed_rule_teacher_v2\\",\\n              \\"review_status\\":\\"unreviewed\\",\\"excluded_ingredients\\":sorted(EXCLUDED),\\n              \\"warning\\":\\"Rule agreement measures imitation, not taste quality or clinical benefit.\\",\\n              \\"splits\\":{s:{\\"records\\":len(rs),\\"independent_queries\\":len({x[\'query_id\'] for x in rs}),\\n                            \\"recipe_families\\":sorted({x[\'recipe_family\'] for x in rs})} for s,rs in rows.items()}}\\n    (directory/\\"manifest.json\\").write_text(json.dumps(manifest,indent=2),encoding=\\"utf-8\\")\\n    return manifest\\n\\ndef read_jsonl(path):\\n    return [json.loads(x) for x in Path(path).read_text(encoding=\\"utf-8\\").splitlines() if x.strip()]\\n\\ndef parse_selection(raw, candidates):\\n    \\"\\"\\"Strict output validation, with no silent \'repair\' in reported model metrics.\\"\\"\\"\\n    try:\\n        obj=json.loads(raw.strip())\\n    except (ValueError,TypeError):\\n        return {\\"json_valid\\":False,\\"candidate_valid\\":False,\\"eligible\\":False,\\"selected\\":None}\\n    exact=isinstance(obj,dict) and set(obj)=={\\"candidate_id\\"}\\n    value=obj.get(\\"candidate_id\\") if isinstance(obj,dict) else None\\n    allowed={c[\\"id\\"]:c for c in candidates}\\n    valid=exact and (value is None or (isinstance(value,str) and value in allowed))\\n    eligible=valid and (choose_teacher(candidates) is None if value is None else allowed[value][\\"eligible\\"])\\n    return {\\"json_valid\\":exact,\\"candidate_valid\\":valid,\\"eligible\\":eligible,\\"selected\\":value}\\n\\ndef summarize(predictions):\\n    n=len(predictions)\\n    return {\\"records\\":n,\\"independent_queries\\":len({p[\\"query_id\\"] for p in predictions}),\\n            \\"recipe_families\\":sorted({p[\\"recipe_family\\"] for p in predictions}),\\n            \\"json_valid_rate\\":sum(p[\\"json_valid\\"] for p in predictions)/n if n else None,\\n            \\"candidate_valid_rate\\":sum(p[\\"candidate_valid\\"] for p in predictions)/n if n else None,\\n            \\"eligible_rate\\":sum(p[\\"eligible\\"] for p in predictions)/n if n else None,\\n            \\"rule_agreement\\":sum(p[\\"candidate_valid\\"] and p[\\"selected\\"]==p[\\"target\\"] for p in predictions)/n if n else None,\\n            \\"interpretation\\":\\"Agreement with unreviewed rules, not independently validated suitability.\\"}\\n\\nif __name__==\\"__main__\\":\\n    print(json.dumps(make_dataset(ROOT/\\"data\\"),indent=2))\\n", "ml/train_lora.py": "\\"\\"\\"Train real LoRA weights and record raw base/adapter metrics. Run on a Colab GPU.\\"\\"\\"\\nfrom __future__ import annotations\\nimport os\\nos.environ.setdefault(\\"USE_TF\\",\\"0\\")\\nos.environ.setdefault(\\"USE_FLAX\\",\\"0\\")\\nimport argparse\\nimport csv\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom datetime import datetime, timezone\\nfrom contextlib import nullcontext\\nfrom pathlib import Path\\nimport torch\\nfrom peft import LoraConfig, TaskType, get_peft_model\\nfrom torch.utils.data import DataLoader, Dataset\\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup\\nfrom policy import ROOT, make_dataset, read_jsonl, parse_selection, summarize, choose_teacher, snapshot_hash\\n\\nBASE_MODEL=\\"Qwen/Qwen2.5-0.5B-Instruct\\"\\n\\nclass Examples(Dataset):\\n    def __init__(self,records,tokenizer,max_length):\\n        self.examples=[]\\n        for record in records:\\n            prefix=tokenizer.apply_chat_template(record[\\"messages\\"],tokenize=False,add_generation_prompt=True)\\n            p=tokenizer.encode(prefix,add_special_tokens=False)\\n            answer=tokenizer.encode(record[\\"completion\\"]+tokenizer.eos_token,add_special_tokens=False)\\n            if len(p)+len(answer)>max_length:\\n                raise ValueError(f\\"{record[\'query_id\']} needs {len(p)+len(answer)} tokens. Increase max_length; do not truncate its answer.\\")\\n            self.examples.append({\\"input_ids\\":p+answer,\\"labels\\":[-100]*len(p)+answer})\\n    def __len__(self):return len(self.examples)\\n    def __getitem__(self,i):return self.examples[i]\\n\\ndef collator(tokenizer):\\n    def collate(rows):\\n        size=max(len(r[\\"input_ids\\"]) for r in rows)\\n        return {\\"input_ids\\":torch.tensor([r[\\"input_ids\\"]+[tokenizer.pad_token_id]*(size-len(r[\\"input_ids\\"])) for r in rows]),\\n                \\"attention_mask\\":torch.tensor([[1]*len(r[\\"input_ids\\"])+[0]*(size-len(r[\\"input_ids\\"])) for r in rows]),\\n                \\"labels\\":torch.tensor([r[\\"labels\\"]+[-100]*(size-len(r[\\"labels\\"])) for r in rows])}\\n    return collate\\n\\ndef move(batch,device):return {k:v.to(device) for k,v in batch.items()}\\n\\n@torch.inference_mode()\\ndef loss_on(model,loader,device):\\n    model.eval()\\n    weighted=0.;tokens=0\\n    for batch in loader:\\n        batch=move(batch,device)\\n        n=int((batch[\\"labels\\"]!=-100).sum())\\n        loss=float(model(**batch).loss)\\n        if not math.isfinite(loss):raise RuntimeError(\\"Nonfinite evaluation loss\\")\\n        weighted+=loss*n;tokens+=n\\n    return weighted/max(tokens,1)\\n\\n@torch.inference_mode()\\ndef generate(model,tokenizer,record,device):\\n    inputs=tokenizer.apply_chat_template(record[\\"messages\\"],add_generation_prompt=True,tokenize=True,\\n                                        return_tensors=\\"pt\\",return_dict=True)\\n    inputs={k:v.to(device) for k,v in inputs.items()}\\n    ids=model.generate(**inputs,max_new_tokens=40,do_sample=False,\\n                       pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)\\n    return tokenizer.decode(ids[0,inputs[\\"input_ids\\"].shape[-1]:],skip_special_tokens=True).strip()\\n\\ndef prediction(model,tokenizer,record,device):\\n    start=time.perf_counter()\\n    raw=generate(model,tokenizer,record,device)\\n    return {\\"query_id\\":record[\\"query_id\\"],\\"recipe_family\\":record[\\"recipe_family\\"],\\n            \\"variant\\":record[\\"variant\\"],\\"target\\":record[\\"target\\"],\\"raw_output\\":raw,\\n            \\"latency_seconds\\":time.perf_counter()-start,**parse_selection(raw,record[\\"candidates\\"])}\\n\\ndef train(args):\\n    if not torch.cuda.is_available() and not args.allow_cpu:\\n        raise RuntimeError(\\"Select Runtime > Change runtime type > T4 GPU in Colab, reconnect and run again.\\")\\n    device=\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n    torch.manual_seed(args.seed);random.seed(args.seed)\\n    if device==\\"cuda\\":torch.cuda.manual_seed_all(args.seed)\\n    args.output.mkdir(parents=True,exist_ok=True)\\n    if not (args.data/\\"train.jsonl\\").exists():make_dataset(args.data)\\n    records={s:read_jsonl(args.data/f\\"{s}.jsonl\\") for s in [\\"train\\",\\"validation\\",\\"test\\"]}\\n    families={s:{x[\\"recipe_family\\"] for x in rs} for s,rs in records.items()}\\n    assert families[\\"train\\"].isdisjoint(families[\\"validation\\"])\\n    assert families[\\"train\\"].isdisjoint(families[\\"test\\"])\\n    assert families[\\"validation\\"].isdisjoint(families[\\"test\\"])\\n    dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device==\\"cuda\\" else torch.float32\\n    tokenizer=AutoTokenizer.from_pretrained(args.model,revision=args.revision)\\n    tokenizer.pad_token=tokenizer.eos_token\\n    tokenizer.padding_side=\\"right\\"\\n    base=AutoModelForCausalLM.from_pretrained(args.model,revision=args.revision,\\n                 torch_dtype=dtype,attn_implementation=\\"sdpa\\").to(device)\\n    revision=getattr(base.config,\\"_commit_hash\\",None)\\n    config=LoraConfig(task_type=TaskType.CAUSAL_LM,r=8,lora_alpha=16,lora_dropout=.05,\\n                      target_modules=[\\"q_proj\\",\\"v_proj\\"],bias=\\"none\\")\\n    model=get_peft_model(base,config)\\n    model.config.use_cache=False\\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\\"use_reentrant\\":False})\\n    trainable={n:p for n,p in model.named_parameters() if p.requires_grad}\\n    assert trainable and all(\\"lora_\\" in n for n in trainable), \\"Only LoRA weights should train\\"\\n    start_weights={n:p.detach().float().cpu().clone() for n,p in trainable.items()}\\n    model.print_trainable_parameters()\\n    loaders={s:DataLoader(Examples(rs,tokenizer,args.max_length),batch_size=args.batch_size,\\n                         shuffle=(s==\\"train\\"),collate_fn=collator(tokenizer)) for s,rs in records.items()}\\n    initial_val=loss_on(model,loaders[\\"validation\\"],device)\\n    optimizer=torch.optim.AdamW(trainable.values(),lr=args.lr,weight_decay=.01)\\n    updates=args.epochs*math.ceil(len(loaders[\\"train\\"])/args.accumulation)\\n    scheduler=get_linear_schedule_with_warmup(optimizer,max(1,int(updates*.1)),updates)\\n    scaler=torch.amp.GradScaler(\\"cuda\\",enabled=(device==\\"cuda\\" and dtype==torch.float16))\\n    logs=[];step=0;best_val=float(\\"inf\\");best_weights=None;started=time.time()\\n    validation=[{\\"epoch\\":0,\\"completion_loss\\":initial_val}]\\n    for epoch in range(args.epochs):\\n        model.train();optimizer.zero_grad(set_to_none=True);running=0.;batches=0\\n        for index,batch in enumerate(loaders[\\"train\\"]):\\n            # Last accumulation window may contain fewer batches.\\n            window_start=(index//args.accumulation)*args.accumulation\\n            window_size=min(args.accumulation,len(loaders[\\"train\\"])-window_start)\\n            ctx=torch.autocast(device_type=\\"cuda\\",dtype=dtype) if device==\\"cuda\\" else nullcontext()\\n            with ctx:loss=model(**move(batch,device)).loss\\n            if not torch.isfinite(loss):raise RuntimeError(\\"Nonfinite training loss. Lower LR or use bf16-capable GPU.\\")\\n            scaler.scale(loss/window_size).backward()\\n            running+=float(loss.detach());batches+=1\\n            if (index+1)%args.accumulation==0 or index+1==len(loaders[\\"train\\"]):\\n                scaler.unscale_(optimizer)\\n                torch.nn.utils.clip_grad_norm_(trainable.values(),1.)\\n                previous_scale=scaler.get_scale()\\n                scaler.step(optimizer);scaler.update()\\n                updated=scaler.get_scale()>=previous_scale\\n                if updated:scheduler.step();step+=1\\n                optimizer.zero_grad(set_to_none=True)\\n                log={\\"optimizer_step\\":step,\\"epoch\\":epoch+1,\\"completion_loss\\":running/batches,\\n                     \\"lr\\":optimizer.param_groups[0][\\"lr\\"],\\"update_applied\\":updated}\\n                logs.append(log);print(json.dumps(log),flush=True);running=0.;batches=0\\n        val=loss_on(model,loaders[\\"validation\\"],device)\\n        validation.append({\\"epoch\\":epoch+1,\\"completion_loss\\":val})\\n        print(\\"VALIDATION\\",validation[-1],flush=True)\\n        if val<best_val:\\n            best_val=val;best_weights={n:p.detach().float().cpu().clone() for n,p in trainable.items()}\\n    with torch.no_grad():\\n        for n,p in trainable.items():p.copy_(best_weights[n].to(device=p.device,dtype=p.dtype))\\n    change=math.sqrt(sum(float(((p.detach().float().cpu()-start_weights[n])**2).sum()) for n,p in trainable.items()))\\n    if not step or not math.isfinite(change) or change<=0:raise RuntimeError(\\"No confirmed LoRA update. Do not report a completed fine-tune.\\")\\n    adapter=args.output/\\"adapter\\"\\n    model.save_pretrained(adapter,safe_serialization=True);tokenizer.save_pretrained(adapter)\\n    model.gradient_checkpointing_disable();model.config.use_cache=True;model.eval()\\n    with (args.output/\\"training_loss.csv\\").open(\\"w\\",newline=\\"\\") as f:\\n        writer=csv.DictWriter(f,fieldnames=list(logs[0]));writer.writeheader();writer.writerows(logs)\\n    (args.output/\\"validation_loss.json\\").write_text(json.dumps(validation,indent=2))\\n    manifest={\\"training_completed\\":True,\\"base_model\\":args.model,\\"base_revision\\":revision,\\n              \\"completed_at_utc\\":datetime.now(timezone.utc).isoformat(),\\n              \\"training_code_sha256\\":hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\\n              \\"adapter_weights_sha256\\":hashlib.sha256((adapter/\\"adapter_model.safetensors\\").read_bytes()).hexdigest(),\\n              \\"seed\\":args.seed,\\"epochs\\":args.epochs,\\"optimizer_steps\\":step,\\n              \\"learning_rate\\":args.lr,\\"lora_rank\\":8,\\"lora_alpha\\":16,\\"target_modules\\":[\\"q_proj\\",\\"v_proj\\"],\\n              \\"trainable_parameters\\":sum(p.numel() for p in trainable.values()),\\n              \\"total_parameters\\":sum(p.numel() for p in model.parameters()),\\n              \\"adapter_weight_l2_change\\":change,\\"snapshot_sha256\\":snapshot_hash(),\\n              \\"data_file_sha256\\":{s:hashlib.sha256((args.data/f\\"{s}.jsonl\\").read_bytes()).hexdigest() for s in records},\\n              \\"split_counts\\":{s:{\\"records\\":len(rs),\\"queries\\":len({x[\'query_id\'] for x in rs}),\\"families\\":sorted(families[s])} for s,rs in records.items()},\\n              \\"label_sources\\":sorted({x.get(\\"label_source\\",\\"unknown\\") for rs in records.values() for x in rs}),\\n              \\"label_review_status\\":sorted({x.get(\\"review_status\\",\\"unknown\\") for rs in records.values() for x in rs}),\\n              \\"initial_validation_completion_loss\\":initial_val,\\"best_validation_completion_loss\\":best_val,\\n              \\"training_seconds\\":time.time()-started,\\"device\\":torch.cuda.get_device_name(0) if device==\\"cuda\\" else \\"CPU\\",\\n              \\"versions\\":{p:importlib.metadata.version(p) for p in [\\"torch\\",\\"transformers\\",\\"peft\\",\\"accelerate\\"]},\\n              \\"interpretation\\":\\"Weakly supervised rule imitation. No independent flavor or suitability accuracy claimed.\\"}\\n    (adapter/\\"training_manifest.json\\").write_text(json.dumps(manifest,indent=2))\\n    if args.skip_generation:\\n        print(\\"Saved trained adapter. Generation evaluation explicitly skipped.\\")\\n        return\\n    comparisons={};predictions={}\\n    for mode in [\\"base\\",\\"lora\\"]:\\n        context=model.disable_adapter() if mode==\\"base\\" else nullcontext()\\n        with context:\\n            predictions[mode]=[prediction(model,tokenizer,r,device) for r in records[\\"test\\"]]\\n        comparisons[mode]=summarize(predictions[mode])\\n    (args.output/\\"evaluation.json\\").write_text(json.dumps(comparisons,indent=2))\\n    (args.output/\\"raw_test_predictions.json\\").write_text(json.dumps(predictions,indent=2))\\n    canonical=read_jsonl(args.data/\\"canonical.jsonl\\")\\n    cache={\\"recorded\\":True,\\"training\\":manifest,\\"test_metrics\\":comparisons,\\"cases\\":{}}\\n    for record in canonical:\\n        entry={\\"split\\":next(s for s,fs in families.items() if record[\\"recipe_family\\"] in fs),\\n               \\"rule_baseline\\":record[\\"target\\"],\\"modes\\":{}}\\n        for mode in [\\"base\\",\\"lora\\"]:\\n            ctx=model.disable_adapter() if mode==\\"base\\" else nullcontext()\\n            with ctx:pred=prediction(model,tokenizer,record,device)\\n            accepted=pred[\\"candidate_valid\\"] and pred[\\"eligible\\"]\\n            selected=pred[\\"selected\\"] if accepted else choose_teacher(record[\\"candidates\\"])\\n            entry[\\"modes\\"][mode]={**pred,\\"recommendation_id\\":selected,\\"used_fallback\\":not accepted}\\n        cache[\\"cases\\"][record[\\"query_id\\"]]=entry\\n    (args.output/\\"lora-demo.json\\").write_text(json.dumps(cache,indent=2))\\n    print(\\"ACTUAL HELD-OUT METRICS\\",json.dumps(comparisons,indent=2),flush=True)\\n    print(\\"Copy lora-demo.json to app/public/. This is a recorded-output demo, not live inference.\\")\\n\\nif __name__==\\"__main__\\":\\n    p=argparse.ArgumentParser()\\n    p.add_argument(\\"--model\\",default=BASE_MODEL);p.add_argument(\\"--revision\\",default=None)\\n    p.add_argument(\\"--data\\",type=Path,default=ROOT/\\"data\\");p.add_argument(\\"--output\\",type=Path,default=ROOT/\\"output\\")\\n    p.add_argument(\\"--epochs\\",type=int,default=3);p.add_argument(\\"--max-length\\",type=int,default=1024)\\n    p.add_argument(\\"--batch-size\\",type=int,default=1);p.add_argument(\\"--accumulation\\",type=int,default=8)\\n    p.add_argument(\\"--lr\\",type=float,default=2e-4);p.add_argument(\\"--seed\\",type=int,default=42)\\n    p.add_argument(\\"--allow-cpu\\",action=\\"store_true\\");p.add_argument(\\"--skip-generation\\",action=\\"store_true\\")\\n    args=p.parse_args()\\n    if min(args.epochs,args.batch_size,args.accumulation,args.max_length)<1:p.error(\\"Epochs and sizes must be positive\\")\\n    train(args)\\n", "ml/api.py": "\\"\\"\\"Optional live local inference. Recorded outputs also work without this server.\\"\\"\\"\\nimport os\\nos.environ.setdefault(\\"USE_TF\\",\\"0\\")\\nos.environ.setdefault(\\"USE_FLAX\\",\\"0\\")\\nimport json\\nimport threading\\nfrom contextlib import nullcontext\\nfrom pathlib import Path\\nfrom typing import Literal\\nfrom fastapi import FastAPI, HTTPException\\nfrom fastapi.middleware.cors import CORSMiddleware\\nfrom pydantic import BaseModel, Field\\nfrom policy import ROOT, build_case, choose_teacher, parse_selection, snapshot_hash\\n\\napp=FastAPI(title=\\"CGAS LoRA Pilot\\")\\napp.add_middleware(CORSMiddleware,allow_origins=[\\n    f\\"http://{host}:{port}\\" for host in [\\"localhost\\",\\"127.0.0.1\\"] for port in [3000,4173,5173]\\n],allow_methods=[\\"GET\\",\\"POST\\"],allow_headers=[\\"Content-Type\\"])\\nADAPTER=Path(os.environ.get(\\"CGAS_ADAPTER_PATH\\",str(ROOT/\\"output\\"/\\"adapter\\")))\\nlock=threading.Lock()\\nruntime=None\\n\\nclass Request(BaseModel):\\n    recipe_id:str=Field(max_length=80)\\n    ingredient_id:str=Field(max_length=80)\\n    goal_id:Literal[\\"satFat\\",\\"calories\\",\\"protein\\",\\"fiber\\"]\\n    mode:Literal[\\"base\\",\\"lora\\"]=\\"lora\\"\\n\\ndef load():\\n    global runtime\\n    if runtime is not None:return runtime\\n    manifest_path=ADAPTER/\\"training_manifest.json\\"\\n    if not manifest_path.exists():raise HTTPException(503,\\"Trained adapter missing. Run the notebook and copy its adapter folder to ml/output/adapter.\\")\\n    manifest=json.loads(manifest_path.read_text())\\n    if not manifest.get(\\"training_completed\\") or manifest.get(\\"adapter_weight_l2_change\\",0)<=0:\\n        raise HTTPException(503,\\"No confirmed LoRA update in the supplied training manifest.\\")\\n    if manifest.get(\\"snapshot_sha256\\")!=snapshot_hash():\\n        raise HTTPException(409,\\"Ingredient snapshot differs from the training run. Use matching files or retrain.\\")\\n    import torch\\n    from transformers import AutoModelForCausalLM, AutoTokenizer\\n    from peft import PeftModel\\n    from train_lora import generate\\n    device=\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n    dtype=torch.float16 if device==\\"cuda\\" else torch.float32\\n    base=AutoModelForCausalLM.from_pretrained(manifest[\\"base_model\\"],revision=manifest.get(\\"base_revision\\"),\\n                                           torch_dtype=dtype,attn_implementation=\\"sdpa\\").to(device)\\n    model=PeftModel.from_pretrained(base,ADAPTER,is_trainable=False)\\n    model.eval()\\n    tokenizer=AutoTokenizer.from_pretrained(ADAPTER)\\n    runtime=(model,tokenizer,device,manifest,generate)\\n    return runtime\\n\\n@app.get(\\"/health\\")\\ndef health():\\n    return {\\"adapter_present\\":(ADAPTER/\\"training_manifest.json\\").exists(),\\"model_loaded\\":runtime is not None}\\n\\n@app.post(\\"/recommend\\")\\ndef recommend(request:Request):\\n    try:case=build_case(request.recipe_id,request.ingredient_id,request.goal_id)\\n    except (ValueError,StopIteration,KeyError):raise HTTPException(400,\\"Unknown recipe, replaceable ingredient or goal.\\")\\n    # Adapter enable/disable is model state. Serialize calls to preserve base/LoRA isolation.\\n    with lock:\\n        model,tokenizer,device,manifest,generate=load()\\n        context=model.disable_adapter() if request.mode==\\"base\\" else nullcontext()\\n        with context:raw=generate(model,tokenizer,case,device)\\n    status=parse_selection(raw,case[\\"candidates\\"])\\n    accepted=status[\\"candidate_valid\\"] and status[\\"eligible\\"]\\n    return {\\"recorded\\":False,\\"mode\\":request.mode,\\"raw_output\\":raw,\\n            \\"rule_baseline\\":case[\\"target\\"],\\"raw_model_candidate\\":status[\\"selected\\"],\\n            \\"recommendation_id\\":status[\\"selected\\"] if accepted else choose_teacher(case[\\"candidates\\"]),\\n            \\"used_fallback\\":not accepted,\\"training\\":manifest,**status}\\n", "ml/test_policy.py": "import json\\nimport tempfile\\nimport unittest\\nfrom pathlib import Path\\nfrom policy import build_case, choose_teacher, make_dataset, parse_selection, read_jsonl, summarize\\n\\nclass PolicyTests(unittest.TestCase):\\n    def test_ineligible_top_candidate_cannot_block_improvement(self):\\n        case=build_case(\\"butter-chicken\\",\\"butter\\",\\"satFat\\")\\n        by_id={x[\\"id\\"]:x for x in case[\\"candidates\\"]}\\n        self.assertFalse(by_id[\\"ghee\\"][\\"eligible\\"])\\n        self.assertTrue(by_id[case[\\"target\\"]][\\"eligible\\"])\\n\\n    def test_zero_fiber_can_improve(self):\\n        case=build_case(\\"butter-chicken\\",\\"cream\\",\\"fiber\\")\\n        self.assertTrue(any(x[\\"eligible\\"] for x in case[\\"candidates\\"]))\\n        self.assertIsNotNone(case[\\"target\\"])\\n        self.assertTrue(all(0<=x[\\"N\\"]<=1 for x in case[\\"candidates\\"]))\\n\\n    def test_null_only_when_no_eligible_candidate(self):\\n        c=build_case(\\"mapo-tofu\\",\\"chiliOil\\",\\"protein\\")\\n        self.assertIsNone(c[\\"target\\"])\\n        self.assertTrue(parse_selection(\'{\\"candidate_id\\":null}\',c[\\"candidates\\"])[\\"eligible\\"])\\n        c=build_case(\\"butter-chicken\\",\\"cream\\",\\"satFat\\")\\n        self.assertFalse(parse_selection(\'{\\"candidate_id\\":null}\',c[\\"candidates\\"])[\\"eligible\\"])\\n\\n    def test_strict_outputs_and_invalid_candidates(self):\\n        case=build_case(\\"butter-chicken\\",\\"cream\\",\\"satFat\\")\\n        for output in [\\"prose\\",\'{\\"candidate_id\\":\\"unknown\\"}\',\'{\\"candidate_id\\":[]}\',\\n                       \'{\\"candidate_id\\":{},\\"extra\\":1}\',\'[]\',\'{\\"candidate_id\\":\\"cashewCream\\",\\"extra\\":1}\']:\\n            self.assertFalse(parse_selection(output,case[\\"candidates\\"])[\\"candidate_valid\\"])\\n        self.assertTrue(parse_selection(case[\\"completion\\"],case[\\"candidates\\"])[\\"eligible\\"])\\n\\n    def test_shuffle_preserves_rule_target(self):\\n        a=build_case(\\"butter-chicken\\",\\"cream\\",\\"satFat\\")\\n        for i in range(12):\\n            b=build_case(\\"butter-chicken\\",\\"cream\\",\\"satFat\\",i)\\n            self.assertEqual(a[\\"target\\"],b[\\"target\\"])\\n\\n    def test_split_family_isolation_and_provenance(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            manifest=make_dataset(directory)\\n            groups=[]\\n            for split in [\\"train\\",\\"validation\\",\\"test\\"]:\\n                rows=read_jsonl(Path(directory)/f\\"{split}.jsonl\\")\\n                groups.append({r[\\"recipe_family\\"] for r in rows})\\n                self.assertTrue(all(r[\\"review_status\\"]==\\"unreviewed\\" for r in rows))\\n                self.assertTrue(all(r[\\"target\\"]==choose_teacher(r[\\"candidates\\"]) for r in rows))\\n                self.assertTrue(all(\\"pecorino\\" not in {c[\\"id\\"] for c in r[\\"candidates\\"]} for r in rows))\\n            for i in range(3):\\n                for j in range(i):self.assertTrue(groups[i].isdisjoint(groups[j]))\\n            self.assertEqual(manifest[\\"splits\\"][\\"test\\"][\\"independent_queries\\"],8)\\n            self.assertEqual(manifest[\\"splits\\"][\\"train\\"][\\"independent_queries\\"],28)\\n\\n    def test_metrics_do_not_hide_invalid_model_outputs(self):\\n        rows=[{\\"query_id\\":\\"q\\",\\"recipe_family\\":\\"r\\",\\"target\\":\\"ok\\",\\"selected\\":None,\\n               \\"candidate_valid\\":False,\\"json_valid\\":False,\\"eligible\\":False}]\\n        self.assertEqual(summarize(rows)[\\"rule_agreement\\"],0)\\n\\nif __name__==\\"__main__\\":unittest.main()\\n", "ml/requirements.txt": "transformers==4.57.3\\npeft==0.18.0\\naccelerate==1.12.0\\nfastapi>=0.115,<1\\nuvicorn>=0.30,<1\\n# Use the CUDA-enabled PyTorch already supplied by Google Colab.\\n", "ml/food_snapshot.json": "{\\n  \\"ingredients\\": {\\n    \\"butter\\": {\\n      \\"id\\": \\"butter\\",\\n      \\"name\\": \\"Butter\\",\\n      \\"category\\": \\"fat \\u00b7 dairy\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 717,\\n        \\"fat\\": 81,\\n        \\"satFat\\": 51,\\n        \\"protein\\": 0.9,\\n        \\"carbs\\": 0.1,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.95,\\n        1,\\n        0.1,\\n        0.2,\\n        0.1,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.9,\\n        \\"italian\\": 0.7,\\n        \\"american\\": 0.85,\\n        \\"chinese\\": 0.15,\\n        \\"mediterranean\\": 0.5\\n      }\\n    },\\n    \\"ghee\\": {\\n      \\"id\\": \\"ghee\\",\\n      \\"name\\": \\"Ghee\\",\\n      \\"category\\": \\"fat \\u00b7 dairy\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 900,\\n        \\"fat\\": 100,\\n        \\"satFat\\": 62,\\n        \\"protein\\": 0,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.85,\\n        0.95,\\n        0.15,\\n        0.15,\\n        0.05,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 1,\\n        \\"italian\\": 0.2,\\n        \\"american\\": 0.3,\\n        \\"chinese\\": 0.1,\\n        \\"mediterranean\\": 0.3\\n      }\\n    },\\n    \\"oliveOil\\": {\\n      \\"id\\": \\"oliveOil\\",\\n      \\"name\\": \\"Olive oil\\",\\n      \\"category\\": \\"fat \\u00b7 plant\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 884,\\n        \\"fat\\": 100,\\n        \\"satFat\\": 14,\\n        \\"protein\\": 0,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.35,\\n        0.55,\\n        0.1,\\n        0.1,\\n        0.15,\\n        0.45\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.35,\\n        \\"italian\\": 0.95,\\n        \\"american\\": 0.6,\\n        \\"chinese\\": 0.2,\\n        \\"mediterranean\\": 1\\n      }\\n    },\\n    \\"avocadoPuree\\": {\\n      \\"id\\": \\"avocadoPuree\\",\\n      \\"name\\": \\"Avocado pur\\u00e9e\\",\\n      \\"category\\": \\"fat \\u00b7 fruit\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 160,\\n        \\"fat\\": 15,\\n        \\"satFat\\": 2.1,\\n        \\"protein\\": 2,\\n        \\"carbs\\": 9,\\n        \\"fiber\\": 6.7\\n      },\\n      \\"flavor\\": [\\n        0.8,\\n        0.55,\\n        0.1,\\n        0.1,\\n        0.1,\\n        0.35\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.25,\\n        \\"italian\\": 0.4,\\n        \\"american\\": 0.7,\\n        \\"chinese\\": 0.1,\\n        \\"mediterranean\\": 0.6\\n      }\\n    },\\n    \\"coconutOil\\": {\\n      \\"id\\": \\"coconutOil\\",\\n      \\"name\\": \\"Coconut oil\\",\\n      \\"category\\": \\"fat \\u00b7 plant\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 892,\\n        \\"fat\\": 100,\\n        \\"satFat\\": 82,\\n        \\"protein\\": 0,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.5,\\n        0.6,\\n        0,\\n        0.35,\\n        0,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.6,\\n        \\"italian\\": 0.1,\\n        \\"american\\": 0.35,\\n        \\"chinese\\": 0.2,\\n        \\"mediterranean\\": 0.2\\n      }\\n    },\\n    \\"cream\\": {\\n      \\"id\\": \\"cream\\",\\n      \\"name\\": \\"Heavy cream\\",\\n      \\"category\\": \\"dairy\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 340,\\n        \\"fat\\": 36,\\n        \\"satFat\\": 23,\\n        \\"protein\\": 2.8,\\n        \\"carbs\\": 2.8,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        1,\\n        0.8,\\n        0.1,\\n        0.25,\\n        0.05,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.85,\\n        \\"italian\\": 0.8,\\n        \\"american\\": 0.8,\\n        \\"chinese\\": 0.1,\\n        \\"mediterranean\\": 0.5\\n      }\\n    },\\n    \\"coconutCream\\": {\\n      \\"id\\": \\"coconutCream\\",\\n      \\"name\\": \\"Coconut cream\\",\\n      \\"category\\": \\"plant cream\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 330,\\n        \\"fat\\": 35,\\n        \\"satFat\\": 31,\\n        \\"protein\\": 3.6,\\n        \\"carbs\\": 6.6,\\n        \\"fiber\\": 2.2\\n      },\\n      \\"flavor\\": [\\n        0.9,\\n        0.5,\\n        0.05,\\n        0.5,\\n        0,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.75,\\n        \\"italian\\": 0.15,\\n        \\"american\\": 0.3,\\n        \\"chinese\\": 0.35,\\n        \\"mediterranean\\": 0.2\\n      }\\n    },\\n    \\"evaporatedMilk\\": {\\n      \\"id\\": \\"evaporatedMilk\\",\\n      \\"name\\": \\"Evaporated milk (2%)\\",\\n      \\"category\\": \\"dairy\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 107,\\n        \\"fat\\": 4,\\n        \\"satFat\\": 2.5,\\n        \\"protein\\": 8,\\n        \\"carbs\\": 11,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.75,\\n        0.45,\\n        0.15,\\n        0.4,\\n        0.1,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.6,\\n        \\"italian\\": 0.5,\\n        \\"american\\": 0.7,\\n        \\"chinese\\": 0.2,\\n        \\"mediterranean\\": 0.4\\n      }\\n    },\\n    \\"greekYogurt\\": {\\n      \\"id\\": \\"greekYogurt\\",\\n      \\"name\\": \\"Greek yogurt (low-fat)\\",\\n      \\"category\\": \\"dairy \\u00b7 fermented\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 73,\\n        \\"fat\\": 1.9,\\n        \\"satFat\\": 1.2,\\n        \\"protein\\": 10,\\n        \\"carbs\\": 3.6,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.7,\\n        0.3,\\n        0.2,\\n        0.15,\\n        0.75,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.85,\\n        \\"italian\\": 0.4,\\n        \\"american\\": 0.6,\\n        \\"chinese\\": 0.15,\\n        \\"mediterranean\\": 0.9\\n      }\\n    },\\n    \\"cashewCream\\": {\\n      \\"id\\": \\"cashewCream\\",\\n      \\"name\\": \\"Cashew cream\\",\\n      \\"category\\": \\"plant cream\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 180,\\n        \\"fat\\": 14,\\n        \\"satFat\\": 2.8,\\n        \\"protein\\": 5.2,\\n        \\"carbs\\": 9.5,\\n        \\"fiber\\": 1.1\\n      },\\n      \\"flavor\\": [\\n        0.9,\\n        0.55,\\n        0.15,\\n        0.3,\\n        0.05,\\n        0.25\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.8,\\n        \\"italian\\": 0.45,\\n        \\"american\\": 0.55,\\n        \\"chinese\\": 0.3,\\n        \\"mediterranean\\": 0.5\\n      }\\n    },\\n    \\"sourCream\\": {\\n      \\"id\\": \\"sourCream\\",\\n      \\"name\\": \\"Sour cream\\",\\n      \\"category\\": \\"dairy \\u00b7 fermented\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 198,\\n        \\"fat\\": 19,\\n        \\"satFat\\": 12,\\n        \\"protein\\": 2.4,\\n        \\"carbs\\": 4.6,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.85,\\n        0.6,\\n        0.15,\\n        0.15,\\n        0.8,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.2,\\n        \\"italian\\": 0.4,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.45\\n      }\\n    },\\n    \\"cremeFraiche\\": {\\n      \\"id\\": \\"cremeFraiche\\",\\n      \\"name\\": \\"Cr\\u00e8me fra\\u00eeche\\",\\n      \\"category\\": \\"dairy \\u00b7 fermented\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 292,\\n        \\"fat\\": 30,\\n        \\"satFat\\": 19,\\n        \\"protein\\": 2.5,\\n        \\"carbs\\": 3.1,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.9,\\n        0.75,\\n        0.1,\\n        0.15,\\n        0.55,\\n        0\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.15,\\n        \\"italian\\": 0.5,\\n        \\"american\\": 0.6,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.55\\n      }\\n    },\\n    \\"pancetta\\": {\\n      \\"id\\": \\"pancetta\\",\\n      \\"name\\": \\"Pancetta\\",\\n      \\"category\\": \\"cured meat\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 407,\\n        \\"fat\\": 35,\\n        \\"satFat\\": 12.5,\\n        \\"protein\\": 22,\\n        \\"carbs\\": 0.5,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.2,\\n        0.75,\\n        0.9,\\n        0.1,\\n        0.05,\\n        0.3\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 1,\\n        \\"american\\": 0.5,\\n        \\"chinese\\": 0.2,\\n        \\"mediterranean\\": 0.7\\n      }\\n    },\\n    \\"turkeyBacon\\": {\\n      \\"id\\": \\"turkeyBacon\\",\\n      \\"name\\": \\"Turkey bacon\\",\\n      \\"category\\": \\"cured meat\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 226,\\n        \\"fat\\": 16,\\n        \\"satFat\\": 4.2,\\n        \\"protein\\": 19,\\n        \\"carbs\\": 1.5,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.1,\\n        0.5,\\n        0.8,\\n        0.1,\\n        0.05,\\n        0.35\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 0.45,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.2,\\n        \\"mediterranean\\": 0.4\\n      }\\n    },\\n    \\"shiitake\\": {\\n      \\"id\\": \\"shiitake\\",\\n      \\"name\\": \\"Roasted shiitake\\",\\n      \\"category\\": \\"vegetable \\u00b7 fungi\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 56,\\n        \\"fat\\": 0.8,\\n        \\"satFat\\": 0.2,\\n        \\"protein\\": 2.2,\\n        \\"carbs\\": 11,\\n        \\"fiber\\": 3.8\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.3,\\n        0.95,\\n        0.05,\\n        0.05,\\n        0.8\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.3,\\n        \\"italian\\": 0.55,\\n        \\"american\\": 0.5,\\n        \\"chinese\\": 0.95,\\n        \\"mediterranean\\": 0.5\\n      }\\n    },\\n    \\"parmesan\\": {\\n      \\"id\\": \\"parmesan\\",\\n      \\"name\\": \\"Parmesan\\",\\n      \\"category\\": \\"cheese \\u00b7 aged\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 431,\\n        \\"fat\\": 29,\\n        \\"satFat\\": 18,\\n        \\"protein\\": 38,\\n        \\"carbs\\": 4.1,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.25,\\n        0.7,\\n        0.95,\\n        0.1,\\n        0.2,\\n        0.35\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 1,\\n        \\"american\\": 0.75,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.85\\n      }\\n    },\\n    \\"nutritionalYeast\\": {\\n      \\"id\\": \\"nutritionalYeast\\",\\n      \\"name\\": \\"Nutritional yeast\\",\\n      \\"category\\": \\"seasoning\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 325,\\n        \\"fat\\": 5,\\n        \\"satFat\\": 0.8,\\n        \\"protein\\": 40,\\n        \\"carbs\\": 35,\\n        \\"fiber\\": 21\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.5,\\n        0.9,\\n        0.05,\\n        0.1,\\n        0.4\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.1,\\n        \\"italian\\": 0.5,\\n        \\"american\\": 0.7,\\n        \\"chinese\\": 0.15,\\n        \\"mediterranean\\": 0.45\\n      }\\n    },\\n    \\"pecorino\\": {\\n      \\"id\\": \\"pecorino\\",\\n      \\"name\\": \\"Pecorino (\\u00bd portion)\\",\\n      \\"category\\": \\"cheese \\u00b7 aged\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 200,\\n        \\"fat\\": 14,\\n        \\"satFat\\": 9,\\n        \\"protein\\": 16,\\n        \\"carbs\\": 2,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.25,\\n        0.65,\\n        0.9,\\n        0.05,\\n        0.3,\\n        0.35\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 0.95,\\n        \\"american\\": 0.5,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.85\\n      }\\n    },\\n    \\"groundPork\\": {\\n      \\"id\\": \\"groundPork\\",\\n      \\"name\\": \\"Ground pork\\",\\n      \\"category\\": \\"meat\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 297,\\n        \\"fat\\": 24,\\n        \\"satFat\\": 8.7,\\n        \\"protein\\": 19,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.2,\\n        0.65,\\n        0.85,\\n        0.05,\\n        0.05,\\n        0.2\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.1,\\n        \\"italian\\": 0.7,\\n        \\"american\\": 0.7,\\n        \\"chinese\\": 0.95,\\n        \\"mediterranean\\": 0.6\\n      }\\n    },\\n    \\"groundTurkey\\": {\\n      \\"id\\": \\"groundTurkey\\",\\n      \\"name\\": \\"Ground turkey (lean)\\",\\n      \\"category\\": \\"meat \\u00b7 poultry\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 149,\\n        \\"fat\\": 8,\\n        \\"satFat\\": 2.1,\\n        \\"protein\\": 20,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.4,\\n        0.7,\\n        0.05,\\n        0.05,\\n        0.15\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.35,\\n        \\"italian\\": 0.55,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.7,\\n        \\"mediterranean\\": 0.6\\n      }\\n    },\\n    \\"tempeh\\": {\\n      \\"id\\": \\"tempeh\\",\\n      \\"name\\": \\"Crumbled tempeh\\",\\n      \\"category\\": \\"plant protein\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 193,\\n        \\"fat\\": 11,\\n        \\"satFat\\": 2.5,\\n        \\"protein\\": 19,\\n        \\"carbs\\": 9.4,\\n        \\"fiber\\": 7\\n      },\\n      \\"flavor\\": [\\n        0.2,\\n        0.35,\\n        0.8,\\n        0.05,\\n        0.1,\\n        0.7\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.3,\\n        \\"italian\\": 0.3,\\n        \\"american\\": 0.55,\\n        \\"chinese\\": 0.75,\\n        \\"mediterranean\\": 0.4\\n      }\\n    },\\n    \\"redLentils\\": {\\n      \\"id\\": \\"redLentils\\",\\n      \\"name\\": \\"Red lentils\\",\\n      \\"category\\": \\"legume\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 116,\\n        \\"fat\\": 0.4,\\n        \\"satFat\\": 0.1,\\n        \\"protein\\": 9,\\n        \\"carbs\\": 20,\\n        \\"fiber\\": 7.9\\n      },\\n      \\"flavor\\": [\\n        0.35,\\n        0.2,\\n        0.4,\\n        0.15,\\n        0.05,\\n        0.75\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.95,\\n        \\"italian\\": 0.35,\\n        \\"american\\": 0.4,\\n        \\"chinese\\": 0.45,\\n        \\"mediterranean\\": 0.7\\n      }\\n    },\\n    \\"caesarDressing\\": {\\n      \\"id\\": \\"caesarDressing\\",\\n      \\"name\\": \\"Caesar dressing\\",\\n      \\"category\\": \\"sauce \\u00b7 emulsion\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 410,\\n        \\"fat\\": 43,\\n        \\"satFat\\": 6.5,\\n        \\"protein\\": 1.6,\\n        \\"carbs\\": 3.3,\\n        \\"fiber\\": 0.2\\n      },\\n      \\"flavor\\": [\\n        0.8,\\n        0.75,\\n        0.6,\\n        0.1,\\n        0.55,\\n        0.15\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 0.5,\\n        \\"american\\": 0.95,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.5\\n      }\\n    },\\n    \\"yogurtCaesar\\": {\\n      \\"id\\": \\"yogurtCaesar\\",\\n      \\"name\\": \\"Yogurt Caesar dressing\\",\\n      \\"category\\": \\"sauce \\u00b7 emulsion\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 130,\\n        \\"fat\\": 9,\\n        \\"satFat\\": 1.8,\\n        \\"protein\\": 5,\\n        \\"carbs\\": 6,\\n        \\"fiber\\": 0.3\\n      },\\n      \\"flavor\\": [\\n        0.7,\\n        0.45,\\n        0.55,\\n        0.1,\\n        0.7,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.15,\\n        \\"italian\\": 0.45,\\n        \\"american\\": 0.85,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.65\\n      }\\n    },\\n    \\"tahiniLemon\\": {\\n      \\"id\\": \\"tahiniLemon\\",\\n      \\"name\\": \\"Tahini\\u2013lemon dressing\\",\\n      \\"category\\": \\"sauce \\u00b7 emulsion\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 220,\\n        \\"fat\\": 19,\\n        \\"satFat\\": 2.6,\\n        \\"protein\\": 6,\\n        \\"carbs\\": 8,\\n        \\"fiber\\": 2.8\\n      },\\n      \\"flavor\\": [\\n        0.65,\\n        0.5,\\n        0.4,\\n        0.05,\\n        0.75,\\n        0.5\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.25,\\n        \\"italian\\": 0.35,\\n        \\"american\\": 0.55,\\n        \\"chinese\\": 0.15,\\n        \\"mediterranean\\": 0.9\\n      }\\n    },\\n    \\"croutons\\": {\\n      \\"id\\": \\"croutons\\",\\n      \\"name\\": \\"Croutons\\",\\n      \\"category\\": \\"grain \\u00b7 baked\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 407,\\n        \\"fat\\": 6.6,\\n        \\"satFat\\": 1.5,\\n        \\"protein\\": 12,\\n        \\"carbs\\": 74,\\n        \\"fiber\\": 4.8\\n      },\\n      \\"flavor\\": [\\n        0.05,\\n        0.4,\\n        0.3,\\n        0.1,\\n        0,\\n        0.5\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.05,\\n        \\"italian\\": 0.6,\\n        \\"american\\": 0.85,\\n        \\"chinese\\": 0.05,\\n        \\"mediterranean\\": 0.6\\n      }\\n    },\\n    \\"toastedChickpeas\\": {\\n      \\"id\\": \\"toastedChickpeas\\",\\n      \\"name\\": \\"Toasted chickpeas\\",\\n      \\"category\\": \\"legume\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 164,\\n        \\"fat\\": 2.6,\\n        \\"satFat\\": 0.3,\\n        \\"protein\\": 8.9,\\n        \\"carbs\\": 27,\\n        \\"fiber\\": 7.6\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.25,\\n        0.4,\\n        0.1,\\n        0.05,\\n        0.7\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.8,\\n        \\"italian\\": 0.4,\\n        \\"american\\": 0.6,\\n        \\"chinese\\": 0.3,\\n        \\"mediterranean\\": 0.9\\n      }\\n    },\\n    \\"chicken\\": {\\n      \\"id\\": \\"chicken\\",\\n      \\"name\\": \\"Chicken\\",\\n      \\"category\\": \\"meat \\u00b7 poultry\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 165,\\n        \\"fat\\": 3.6,\\n        \\"satFat\\": 1,\\n        \\"protein\\": 31,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.45,\\n        0.75,\\n        0.05,\\n        0.05,\\n        0.15\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.95,\\n        \\"italian\\": 0.7,\\n        \\"american\\": 0.95,\\n        \\"chinese\\": 0.9,\\n        \\"mediterranean\\": 0.85\\n      }\\n    },\\n    \\"tomato\\": {\\n      \\"id\\": \\"tomato\\",\\n      \\"name\\": \\"Tomato\\",\\n      \\"category\\": \\"vegetable \\u00b7 fruit\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 18,\\n        \\"fat\\": 0.2,\\n        \\"satFat\\": 0,\\n        \\"protein\\": 0.9,\\n        \\"carbs\\": 3.9,\\n        \\"fiber\\": 1.2\\n      },\\n      \\"flavor\\": [\\n        0.05,\\n        0.05,\\n        0.55,\\n        0.45,\\n        0.85,\\n        0.2\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.9,\\n        \\"italian\\": 1,\\n        \\"american\\": 0.8,\\n        \\"chinese\\": 0.4,\\n        \\"mediterranean\\": 0.95\\n      }\\n    },\\n    \\"onion\\": {\\n      \\"id\\": \\"onion\\",\\n      \\"name\\": \\"Onion\\",\\n      \\"category\\": \\"vegetable \\u00b7 allium\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 40,\\n        \\"fat\\": 0.1,\\n        \\"satFat\\": 0,\\n        \\"protein\\": 1.1,\\n        \\"carbs\\": 9.3,\\n        \\"fiber\\": 1.7\\n      },\\n      \\"flavor\\": [\\n        0.05,\\n        0.1,\\n        0.6,\\n        0.4,\\n        0.2,\\n        0.5\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.95,\\n        \\"italian\\": 0.85,\\n        \\"american\\": 0.85,\\n        \\"chinese\\": 0.8,\\n        \\"mediterranean\\": 0.9\\n      }\\n    },\\n    \\"garlic\\": {\\n      \\"id\\": \\"garlic\\",\\n      \\"name\\": \\"Garlic\\",\\n      \\"category\\": \\"vegetable \\u00b7 allium\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 149,\\n        \\"fat\\": 0.5,\\n        \\"satFat\\": 0.1,\\n        \\"protein\\": 6.4,\\n        \\"carbs\\": 33,\\n        \\"fiber\\": 2.1\\n      },\\n      \\"flavor\\": [\\n        0.05,\\n        0.1,\\n        0.85,\\n        0.1,\\n        0.15,\\n        0.85\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.95,\\n        \\"italian\\": 0.95,\\n        \\"american\\": 0.8,\\n        \\"chinese\\": 0.95,\\n        \\"mediterranean\\": 0.95\\n      }\\n    },\\n    \\"ginger\\": {\\n      \\"id\\": \\"ginger\\",\\n      \\"name\\": \\"Ginger\\",\\n      \\"category\\": \\"spice \\u00b7 rhizome\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 80,\\n        \\"fat\\": 0.8,\\n        \\"satFat\\": 0.2,\\n        \\"protein\\": 1.8,\\n        \\"carbs\\": 18,\\n        \\"fiber\\": 2\\n      },\\n      \\"flavor\\": [\\n        0,\\n        0.05,\\n        0.2,\\n        0.2,\\n        0.4,\\n        0.95\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.95,\\n        \\"italian\\": 0.1,\\n        \\"american\\": 0.4,\\n        \\"chinese\\": 0.95,\\n        \\"mediterranean\\": 0.3\\n      }\\n    },\\n    \\"garamMasala\\": {\\n      \\"id\\": \\"garamMasala\\",\\n      \\"name\\": \\"Garam masala\\",\\n      \\"category\\": \\"spice blend\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 325,\\n        \\"fat\\": 12,\\n        \\"satFat\\": 2,\\n        \\"protein\\": 13,\\n        \\"carbs\\": 48,\\n        \\"fiber\\": 18\\n      },\\n      \\"flavor\\": [\\n        0,\\n        0.1,\\n        0.3,\\n        0.25,\\n        0.1,\\n        1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 1,\\n        \\"italian\\": 0.05,\\n        \\"american\\": 0.2,\\n        \\"chinese\\": 0.15,\\n        \\"mediterranean\\": 0.25\\n      }\\n    },\\n    \\"pasta\\": {\\n      \\"id\\": \\"pasta\\",\\n      \\"name\\": \\"Spaghetti (dry)\\",\\n      \\"category\\": \\"grain\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 371,\\n        \\"fat\\": 1.5,\\n        \\"satFat\\": 0.3,\\n        \\"protein\\": 13,\\n        \\"carbs\\": 74,\\n        \\"fiber\\": 3.2\\n      },\\n      \\"flavor\\": [\\n        0.2,\\n        0.2,\\n        0.2,\\n        0.1,\\n        0,\\n        0.2\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.1,\\n        \\"italian\\": 1,\\n        \\"american\\": 0.8,\\n        \\"chinese\\": 0.3,\\n        \\"mediterranean\\": 0.8\\n      }\\n    },\\n    \\"egg\\": {\\n      \\"id\\": \\"egg\\",\\n      \\"name\\": \\"Eggs\\",\\n      \\"category\\": \\"protein \\u00b7 egg\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 155,\\n        \\"fat\\": 11,\\n        \\"satFat\\": 3.3,\\n        \\"protein\\": 13,\\n        \\"carbs\\": 1.1,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.7,\\n        0.65,\\n        0.5,\\n        0.05,\\n        0.05,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.6,\\n        \\"italian\\": 0.85,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.85,\\n        \\"mediterranean\\": 0.8\\n      }\\n    },\\n    \\"blackPepper\\": {\\n      \\"id\\": \\"blackPepper\\",\\n      \\"name\\": \\"Black pepper\\",\\n      \\"category\\": \\"spice\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 251,\\n        \\"fat\\": 3.3,\\n        \\"satFat\\": 1.4,\\n        \\"protein\\": 10,\\n        \\"carbs\\": 64,\\n        \\"fiber\\": 25\\n      },\\n      \\"flavor\\": [\\n        0,\\n        0.05,\\n        0.2,\\n        0,\\n        0.1,\\n        1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.85,\\n        \\"italian\\": 0.9,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.7,\\n        \\"mediterranean\\": 0.85\\n      }\\n    },\\n    \\"beef\\": {\\n      \\"id\\": \\"beef\\",\\n      \\"name\\": \\"Beef sirloin\\",\\n      \\"category\\": \\"meat\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 183,\\n        \\"fat\\": 8.5,\\n        \\"satFat\\": 3.4,\\n        \\"protein\\": 27,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.7,\\n        0.95,\\n        0,\\n        0.05,\\n        0.35\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.2,\\n        \\"italian\\": 0.6,\\n        \\"american\\": 0.95,\\n        \\"chinese\\": 0.7,\\n        \\"mediterranean\\": 0.6\\n      }\\n    },\\n    \\"mushroom\\": {\\n      \\"id\\": \\"mushroom\\",\\n      \\"name\\": \\"Mushrooms\\",\\n      \\"category\\": \\"vegetable \\u00b7 fungi\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 22,\\n        \\"fat\\": 0.3,\\n        \\"satFat\\": 0.1,\\n        \\"protein\\": 3.1,\\n        \\"carbs\\": 3.3,\\n        \\"fiber\\": 1\\n      },\\n      \\"flavor\\": [\\n        0.2,\\n        0.25,\\n        0.9,\\n        0.05,\\n        0.05,\\n        0.75\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.4,\\n        \\"italian\\": 0.8,\\n        \\"american\\": 0.8,\\n        \\"chinese\\": 0.9,\\n        \\"mediterranean\\": 0.75\\n      }\\n    },\\n    \\"flour\\": {\\n      \\"id\\": \\"flour\\",\\n      \\"name\\": \\"Flour\\",\\n      \\"category\\": \\"grain\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 364,\\n        \\"fat\\": 1,\\n        \\"satFat\\": 0.2,\\n        \\"protein\\": 10,\\n        \\"carbs\\": 76,\\n        \\"fiber\\": 2.7\\n      },\\n      \\"flavor\\": [\\n        0.15,\\n        0.1,\\n        0.05,\\n        0.1,\\n        0,\\n        0.1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.8,\\n        \\"italian\\": 0.9,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.7,\\n        \\"mediterranean\\": 0.8\\n      }\\n    },\\n    \\"tofu\\": {\\n      \\"id\\": \\"tofu\\",\\n      \\"name\\": \\"Silken tofu\\",\\n      \\"category\\": \\"plant protein\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 55,\\n        \\"fat\\": 2.7,\\n        \\"satFat\\": 0.4,\\n        \\"protein\\": 5,\\n        \\"carbs\\": 2.9,\\n        \\"fiber\\": 0.3\\n      },\\n      \\"flavor\\": [\\n        0.75,\\n        0.25,\\n        0.35,\\n        0.1,\\n        0.05,\\n        0.2\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.2,\\n        \\"italian\\": 0.1,\\n        \\"american\\": 0.4,\\n        \\"chinese\\": 1,\\n        \\"mediterranean\\": 0.2\\n      }\\n    },\\n    \\"doubanjiang\\": {\\n      \\"id\\": \\"doubanjiang\\",\\n      \\"name\\": \\"Doubanjiang\\",\\n      \\"category\\": \\"paste \\u00b7 fermented\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 130,\\n        \\"fat\\": 5,\\n        \\"satFat\\": 0.8,\\n        \\"protein\\": 8,\\n        \\"carbs\\": 14,\\n        \\"fiber\\": 3\\n      },\\n      \\"flavor\\": [\\n        0.1,\\n        0.3,\\n        0.95,\\n        0.15,\\n        0.2,\\n        0.9\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.1,\\n        \\"italian\\": 0.05,\\n        \\"american\\": 0.2,\\n        \\"chinese\\": 1,\\n        \\"mediterranean\\": 0.1\\n      }\\n    },\\n    \\"chiliOil\\": {\\n      \\"id\\": \\"chiliOil\\",\\n      \\"name\\": \\"Sichuan chili oil\\",\\n      \\"category\\": \\"fat \\u00b7 infused\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 884,\\n        \\"fat\\": 100,\\n        \\"satFat\\": 15,\\n        \\"protein\\": 0,\\n        \\"carbs\\": 0,\\n        \\"fiber\\": 0\\n      },\\n      \\"flavor\\": [\\n        0.1,\\n        0.4,\\n        0.5,\\n        0,\\n        0.05,\\n        1\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.4,\\n        \\"italian\\": 0.1,\\n        \\"american\\": 0.3,\\n        \\"chinese\\": 1,\\n        \\"mediterranean\\": 0.15\\n      }\\n    },\\n    \\"scallion\\": {\\n      \\"id\\": \\"scallion\\",\\n      \\"name\\": \\"Scallion\\",\\n      \\"category\\": \\"vegetable \\u00b7 allium\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 32,\\n        \\"fat\\": 0.2,\\n        \\"satFat\\": 0,\\n        \\"protein\\": 1.8,\\n        \\"carbs\\": 7.3,\\n        \\"fiber\\": 2.6\\n      },\\n      \\"flavor\\": [\\n        0.05,\\n        0.05,\\n        0.4,\\n        0.2,\\n        0.3,\\n        0.6\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.5,\\n        \\"italian\\": 0.3,\\n        \\"american\\": 0.5,\\n        \\"chinese\\": 0.95,\\n        \\"mediterranean\\": 0.4\\n      }\\n    },\\n    \\"romaine\\": {\\n      \\"id\\": \\"romaine\\",\\n      \\"name\\": \\"Romaine lettuce\\",\\n      \\"category\\": \\"vegetable \\u00b7 leafy\\",\\n      \\"per100g\\": {\\n        \\"calories\\": 17,\\n        \\"fat\\": 0.3,\\n        \\"satFat\\": 0,\\n        \\"protein\\": 1.2,\\n        \\"carbs\\": 3.3,\\n        \\"fiber\\": 2.1\\n      },\\n      \\"flavor\\": [\\n        0.1,\\n        0.05,\\n        0.1,\\n        0.15,\\n        0.1,\\n        0.3\\n      ],\\n      \\"cuisine\\": {\\n        \\"indian\\": 0.1,\\n        \\"italian\\": 0.6,\\n        \\"american\\": 0.9,\\n        \\"chinese\\": 0.3,\\n        \\"mediterranean\\": 0.8\\n      }\\n    }\\n  },\\n  \\"recipes\\": [\\n    {\\n      \\"id\\": \\"butter-chicken\\",\\n      \\"name\\": \\"Butter Chicken\\",\\n      \\"cuisine\\": \\"indian\\",\\n      \\"serves\\": 4,\\n      \\"note\\": \\"Murgh makhani \\u2014 tomato, butter & cream\\",\\n      \\"items\\": [\\n        {\\n          \\"ingredientId\\": \\"chicken\\",\\n          \\"grams\\": 500\\n        },\\n        {\\n          \\"ingredientId\\": \\"butter\\",\\n          \\"grams\\": 60,\\n          \\"replaceable\\": [\\n            \\"ghee\\",\\n            \\"oliveOil\\",\\n            \\"avocadoPuree\\",\\n            \\"coconutOil\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"cream\\",\\n          \\"grams\\": 120,\\n          \\"replaceable\\": [\\n            \\"coconutCream\\",\\n            \\"evaporatedMilk\\",\\n            \\"greekYogurt\\",\\n            \\"cashewCream\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"tomato\\",\\n          \\"grams\\": 200\\n        },\\n        {\\n          \\"ingredientId\\": \\"onion\\",\\n          \\"grams\\": 150\\n        },\\n        {\\n          \\"ingredientId\\": \\"garlic\\",\\n          \\"grams\\": 15\\n        },\\n        {\\n          \\"ingredientId\\": \\"ginger\\",\\n          \\"grams\\": 15\\n        },\\n        {\\n          \\"ingredientId\\": \\"garamMasala\\",\\n          \\"grams\\": 10\\n        }\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"carbonara\\",\\n      \\"name\\": \\"Pasta Carbonara\\",\\n      \\"cuisine\\": \\"italian\\",\\n      \\"serves\\": 4,\\n      \\"note\\": \\"Roman classic \\u2014 egg, pecorino & pancetta\\",\\n      \\"items\\": [\\n        {\\n          \\"ingredientId\\": \\"pasta\\",\\n          \\"grams\\": 320\\n        },\\n        {\\n          \\"ingredientId\\": \\"pancetta\\",\\n          \\"grams\\": 150,\\n          \\"replaceable\\": [\\n            \\"turkeyBacon\\",\\n            \\"shiitake\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"egg\\",\\n          \\"grams\\": 200\\n        },\\n        {\\n          \\"ingredientId\\": \\"parmesan\\",\\n          \\"grams\\": 60,\\n          \\"replaceable\\": [\\n            \\"nutritionalYeast\\",\\n            \\"pecorino\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"blackPepper\\",\\n          \\"grams\\": 4\\n        }\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"stroganoff\\",\\n      \\"name\\": \\"Beef Stroganoff\\",\\n      \\"cuisine\\": \\"american\\",\\n      \\"serves\\": 4,\\n      \\"note\\": \\"Sour-cream pan sauce with mushrooms\\",\\n      \\"items\\": [\\n        {\\n          \\"ingredientId\\": \\"beef\\",\\n          \\"grams\\": 500\\n        },\\n        {\\n          \\"ingredientId\\": \\"butter\\",\\n          \\"grams\\": 40,\\n          \\"replaceable\\": [\\n            \\"ghee\\",\\n            \\"oliveOil\\",\\n            \\"avocadoPuree\\",\\n            \\"coconutOil\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"sourCream\\",\\n          \\"grams\\": 200,\\n          \\"replaceable\\": [\\n            \\"greekYogurt\\",\\n            \\"cremeFraiche\\",\\n            \\"cashewCream\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"mushroom\\",\\n          \\"grams\\": 200\\n        },\\n        {\\n          \\"ingredientId\\": \\"onion\\",\\n          \\"grams\\": 150\\n        },\\n        {\\n          \\"ingredientId\\": \\"flour\\",\\n          \\"grams\\": 20\\n        }\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"mapo-tofu\\",\\n      \\"name\\": \\"Mapo Tofu\\",\\n      \\"cuisine\\": \\"chinese\\",\\n      \\"serves\\": 4,\\n      \\"note\\": \\"Sichuan \\u2014 numbing chili-bean sauce\\",\\n      \\"items\\": [\\n        {\\n          \\"ingredientId\\": \\"tofu\\",\\n          \\"grams\\": 400\\n        },\\n        {\\n          \\"ingredientId\\": \\"groundPork\\",\\n          \\"grams\\": 150,\\n          \\"replaceable\\": [\\n            \\"groundTurkey\\",\\n            \\"tempeh\\",\\n            \\"redLentils\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"doubanjiang\\",\\n          \\"grams\\": 45\\n        },\\n        {\\n          \\"ingredientId\\": \\"chiliOil\\",\\n          \\"grams\\": 30,\\n          \\"replaceable\\": [\\n            \\"oliveOil\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"garlic\\",\\n          \\"grams\\": 10\\n        },\\n        {\\n          \\"ingredientId\\": \\"scallion\\",\\n          \\"grams\\": 20\\n        }\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"caesar\\",\\n      \\"name\\": \\"Caesar Salad\\",\\n      \\"cuisine\\": \\"american\\",\\n      \\"serves\\": 4,\\n      \\"note\\": \\"Emulsified dressing, parmesan, crunch\\",\\n      \\"items\\": [\\n        {\\n          \\"ingredientId\\": \\"romaine\\",\\n          \\"grams\\": 300\\n        },\\n        {\\n          \\"ingredientId\\": \\"caesarDressing\\",\\n          \\"grams\\": 120,\\n          \\"replaceable\\": [\\n            \\"yogurtCaesar\\",\\n            \\"tahiniLemon\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"parmesan\\",\\n          \\"grams\\": 40,\\n          \\"replaceable\\": [\\n            \\"nutritionalYeast\\",\\n            \\"pecorino\\"\\n          ]\\n        },\\n        {\\n          \\"ingredientId\\": \\"croutons\\",\\n          \\"grams\\": 60,\\n          \\"replaceable\\": [\\n            \\"toastedChickpeas\\"\\n          ]\\n        }\\n      ]\\n    }\\n  ],\\n  \\"goals\\": [\\n    {\\n      \\"id\\": \\"satFat\\",\\n      \\"label\\": \\"Reduce saturated fat\\",\\n      \\"unit\\": \\"g\\",\\n      \\"direction\\": \\"min\\"\\n    },\\n    {\\n      \\"id\\": \\"calories\\",\\n      \\"label\\": \\"Reduce calories\\",\\n      \\"unit\\": \\"kcal\\",\\n      \\"direction\\": \\"min\\"\\n    },\\n    {\\n      \\"id\\": \\"protein\\",\\n      \\"label\\": \\"Increase protein\\",\\n      \\"unit\\": \\"g\\",\\n      \\"direction\\": \\"max\\"\\n    },\\n    {\\n      \\"id\\": \\"fiber\\",\\n      \\"label\\": \\"Increase fiber\\",\\n      \\"unit\\": \\"g\\",\\n      \\"direction\\": \\"max\\"\\n    }\\n  ],\\n  \\"weights\\": {\\n    \\"flavor\\": 0.35,\\n    \\"nutrition\\": 0.3,\\n    \\"cuisine\\": 0.25,\\n    \\"category\\": 0.1\\n  },\\n  \\"source\\": \\"User-supplied Kimi recipe app. Illustrative, unverified data.\\"\\n}"}')
for relative, content in BUNDLE.items():
    target = TASK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
print('Wrote', len(BUNDLE), 'files to', TASK)


## 3. Install pinned text libraries

Use Colab's existing CUDA-enabled PyTorch. Training runs in a new Python process, avoiding notebook import caching.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(TASK/'ml/requirements.txt')], check=True)


## 4. Generate and inspect the training data

Training uses 28 independent queries from three recipe families, with eight
candidate-order variants each. Validation and testing each use eight queries
from one other recipe family, with three order variants. These variants are
augmentation, not new independent examples. Some ingredients occur in multiple
partitions, so this does not test wholly unseen ingredients.

If you have reviewed data, replace the labels and document the real provenance
before training. This default run deliberately labels its data as rule-derived.


In [ ]:
subprocess.run([sys.executable, 'ml/policy.py'], cwd=TASK, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'ml', '-p', 'test_*.py', '-v'], cwd=TASK, check=True)
dataset_manifest = json.loads((TASK/'ml/data/manifest.json').read_text())
sample = json.loads((TASK/'ml/data/train.jsonl').read_text().splitlines()[0])
print(json.dumps(dataset_manifest, indent=2))
print('Example prompt:', sample['messages'])
print('Rule-derived target:', sample['completion'])


## 5. Fine-tune the LoRA adapter

The pretrained model stays frozen. Only rank-8 adapters on the attention query
and value projections train. The loss ignores prompt tokens and supervises
the answer. Validation loss selects the best epoch. Raw test outputs compare
the same base model with its adapter enabled and disabled.

Keep the defaults for the first run. Training, downloads and generation take
time depending on GPU availability. The process prints real progress. A GPU
memory error can be addressed by keeping batch size at 1 and checking sequence
lengths before reducing max length. Do not truncate supervised answers.


In [ ]:
subprocess.run([
    sys.executable, 'ml/train_lora.py',
    '--epochs', '3', '--max-length', '1024',
    '--batch-size', '1', '--accumulation', '8',
], cwd=TASK, check=True)


## 6. Inspect actual training evidence and test results

In [ ]:
OUT = TASK/'ml/output'
manifest = json.loads((OUT/'adapter/training_manifest.json').read_text())
metrics = json.loads((OUT/'evaluation.json').read_text())
assert manifest['training_completed'] and manifest['adapter_weight_l2_change'] > 0
assert manifest['optimizer_steps'] > 0
print(json.dumps(manifest, indent=2))
print('ACTUAL RAW-OUTPUT METRICS:')
print(json.dumps(metrics, indent=2))
print('These measure agreement with unreviewed rules, not expert suitability.')


In [ ]:
import csv
import matplotlib.pyplot as plt
logs = list(csv.DictReader((OUT/'training_loss.csv').open()))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot([int(r['optimizer_step']) for r in logs], [float(r['completion_loss']) for r in logs])
ax.set(xlabel='Optimizer update', ylabel='Completion loss', title='Actual LoRA training loss')
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUT/'training_loss.png', dpi=160)
plt.show()


## 7. Export the run for your presentation

The ZIP includes the adapter, logs, raw predictions, metric table and recorded
website outputs. The recorded demo file comes from model generation after
training. It is not a substitute for a live inference claim.


In [ ]:
def pct(x):
    return 'N/A' if x is None else f'{x*100:.1f}%'
text = '# Actual CGAS LoRA run\n\n'
text += f"Base model: {manifest['base_model']}\n\n"
text += f"Resolved revision: {manifest['base_revision']}\n\n"
text += f"Trainable parameters: {manifest['trainable_parameters']:,} / {manifest['total_parameters']:,} "
text += f"({100*manifest['trainable_parameters']/manifest['total_parameters']:.4f}%).\n\n"
text += f"LoRA rank 8, alpha 16, q_proj/v_proj, {manifest['epochs']} epochs, {manifest['optimizer_steps']} optimizer updates.\n\n"
text += f"Adapter weight L2 change: {manifest['adapter_weight_l2_change']:.6f}.\n\n"
text += 'Labels: unreviewed fixed-rule weak supervision.\n\n'
text += '| Metric | Base | LoRA |\n|---|---:|---:|\n'
for key in ['json_valid_rate','candidate_valid_rate','eligible_rate','rule_agreement']:
    text += f"| {key} | {pct(metrics['base'][key])} | {pct(metrics['lora'][key])} |\n"
text += f"\nTest records: {metrics['lora']['records']}. Independent queries: {metrics['lora']['independent_queries']}. "
text += f"Recipe families: {metrics['lora']['recipe_families']}.\n\n"
text += 'Do not call this taste accuracy or claim superiority if the actual comparison does not show it.\n'
(OUT/'PRESENTATION_NUMBERS.md').write_text(text)
print(text)
import zipfile
export_path = Path('/content/CGAS_LoRA_Run.zip')
with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file(): archive.write(path, path.relative_to(OUT))
    for path in (TASK/'ml/data').rglob('*'):
        if path.is_file(): archive.write(path, Path('dataset')/path.relative_to(TASK/'ml/data'))
print('Run saved:', export_path)
from google.colab import files
files.download(str(export_path))


## 8. Connect the results to the website

Extract `CGAS_LoRA_Run.zip`. Copy `lora-demo.json` to your app's `public` folder.
Use the **LoRA pilot comparison** panel and its recorded-output mode. Choose
the base model and adapter to show the comparison.

For optional live inference, also copy the `adapter` folder to `ml/output/adapter`
and start the server described in `README_LORA.md`. Base model weights download
on the first live request if not already cached. The small adapter alone does
not contain the entire base model.

For the presentation, use the measured counts and metrics in
`PRESENTATION_NUMBERS.md`. Completing a fine-tune does not guarantee improved
held-out predictions. Replacing the weak labels with independently reviewed
substitutions is the next scientific improvement.
